In [1]:
import pandas as pd
import numpy as np
from sklearn.ensemble import RandomForestClassifier
from sklearn.preprocessing import LabelEncoder
from sklearn.model_selection import train_test_split

# 1. Load Data
train_df = pd.read_csv('train.csv')
test_df = pd.read_csv('test.csv')

# 2. Preprocessing & Model Training (Quick Re-run)
# Drop IDs for training
X_train = train_df.drop(['id', 'CustomerId', 'Surname', 'Exited'], axis=1)
y_train = train_df['Exited']
X_test = test_df.drop(['id', 'CustomerId', 'Surname'], axis=1)

# Encoding
le = LabelEncoder()
X_train['Gender'] = le.fit_transform(X_train['Gender'])
X_test['Gender'] = le.transform(X_test['Gender'])

X_train = pd.get_dummies(X_train, columns=['Geography'], drop_first=True)
X_test = pd.get_dummies(X_test, columns=['Geography'], drop_first=True)

# Train Model
rf = RandomForestClassifier(n_estimators=100, random_state=42)
rf.fit(X_train, y_train)

# 3. Generate Predictions
probs = rf.predict_proba(X_test)[:, 1]

# 4. Create the "Dashboard Ready" Dataset
# We take the original Test data (with IDs/Names) and add our insights
dashboard_df = test_df.copy()
dashboard_df['Churn_Probability'] = probs

# Add Risk Segment
conditions = [
    (dashboard_df['Churn_Probability'] >= 0.8),
    (dashboard_df['Churn_Probability'] >= 0.5) & (dashboard_df['Churn_Probability'] < 0.8),
    (dashboard_df['Churn_Probability'] < 0.5)
]
choices = ['High Risk 🔴', 'Medium Risk 🟡', 'Low Risk 🟢']
dashboard_df['Risk_Category'] = np.select(conditions, choices, default='Low Risk 🟢')

# 5. Simulate AI Insights (Rule-Based Generation for Dashboard)
# In a real app, this comes from the API. For the CSV, we simulate it based on logic.
def generate_ai_email(row):
    if row['Churn_Probability'] < 0.5:
        return "N/A"
    
    email = f"Dear {row['Surname']},\n\n"
    if row['Balance'] > 100000:
        email += f"As a premium member with ${row['Balance']:,.0f}, we'd like to offer you our Wealth Advisory service..."
    elif row['NumOfProducts'] >= 3:
        email += "We noticed you have multiple accounts. Let's consolidate them for better rates..."
    else:
        email += "We miss you! Here is a 2% cashback offer on your next transaction..."
    return email

dashboard_df['AI_Generated_Email'] = dashboard_df.apply(generate_ai_email, axis=1)

# Save to CSV
dashboard_df.to_csv('Bank360_Dashboard_Data.csv', index=False)

print("Dashboard Data Generated: 'Bank360_Dashboard_Data.csv'")
print(dashboard_df[['Surname', 'Churn_Probability', 'Risk_Category', 'AI_Generated_Email']].head())

Dashboard Data Generated: 'Bank360_Dashboard_Data.csv'
     Surname  Churn_Probability Risk_Category  \
0   Lucchese               0.02    Low Risk 🟢   
1       Nott               0.90   High Risk 🔴   
2         K?               0.00    Low Risk 🟢   
3  O'Donnell               0.12    Low Risk 🟢   
4    Higgins               0.27    Low Risk 🟢   

                                  AI_Generated_Email  
0                                                N/A  
1  Dear Nott,\n\nWe miss you! Here is a 2% cashba...  
2                                                N/A  
3                                                N/A  
4                                                N/A  
